In [37]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import backend.config as config

In [38]:
ticker = "MSFT"
interval = "daily"
start_date_string = "2026-01-01"
end_date_string = None

bronze_path = f"../../../data/bronze/{ticker}/{interval}/news.parquet"
silver_path = f"../../../data/silver/{ticker}/{interval}/news.parquet"
gold_path = f"../../../data/gold/{ticker}/{interval}/baseline_results.parquet"

In [39]:
df_news = pd.read_parquet(bronze_path, engine='pyarrow').reset_index(drop=True)
df_news['Date'] = pd.to_datetime(df_news['created_at']).dt.tz_localize(None).dt.normalize()
print(f"News DataFrame shape: {df_news.shape}")
df_news.sort_values(by='created_at', ascending=False).head()

News DataFrame shape: (611, 4)


,headline,summary,created_at,Date
0,"Trump's Trade Wars, AI Displacement, Bitcoin's...",It’s been an eventful week in the world of bus...,2026-03-08 10:00:53+00:00,2026-03-08
1,Engineer Describes 19-Hour Workday at Elon Mus...,xAI engineer Giri Kuncoro&#39;s viral posts sp...,2026-03-07 17:31:29+00:00,2026-03-07
2,Bill Gates Once Revealed A Vital Personality T...,Bill Gates emphasizes the importance of buildi...,2026-03-07 15:30:16+00:00,2026-03-07
3,Pentagon Tech Chief Emil Michael Slams Anthrop...,Pentagon tech chief Emil Michael criticized An...,2026-03-07 06:31:27+00:00,2026-03-07
4,"Amazon, Google And Microsoft Keep Anthropic AI...",Amazon and Google will continue offering Anthr...,2026-03-07 04:49:26+00:00,2026-03-07


In [40]:
df_news['headline'][0]

"Trump's Trade Wars, AI Displacement, Bitcoin's Future And More: This Week In Economy"

In [41]:
df_news['summary'][0]

'It’s been an eventful week in the world of business and finance. Here’s a quick look at the top stories that made headlines.'

In [42]:
grouped_news = df_news.groupby('Date')
grouped_news.size()

Date
2026-01-01     4
2026-01-02    14
2026-01-05     5
2026-01-06     6
2026-01-07     9
              ..
2026-03-04    10
2026-03-05    17
2026-03-06    12
2026-03-07     4
2026-03-08     1
Length: 61, dtype: int64

In [43]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate



llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=config.OPENAI_KEY)
template = """
Analyze these headlines for {ticker}. 
Score the narrative from -1.0 (Panic/Crisis) to 1.0 (Euphoria/Growth).
Headlines: {news}
Return ONLY the numerical score.
"""
prompt = PromptTemplate.from_template(template)

daily_scores = []

print(f"Computing LLM Sentiment across {len(grouped_news)} trading days...")

for date, group in grouped_news:
    headlines = "\n- ".join(group['headline'].dropna().tolist())

    print(f"Date: {date.date()}, Headlines: {len(group)}")

    

    response = llm.invoke(prompt.format(ticker=ticker, news=headlines))
    score = float(response.content.strip())

    
    daily_scores.append({'Date': date, 'Sentiment': score})

    break
    
sentiment_df = pd.DataFrame(daily_scores).set_index('Date')

Computing LLM Sentiment across 61 trading days...
Date: 2026-01-01, Headlines: 4


In [44]:
sentiment_df

,Sentiment
Date,
2026-01-01,0.0


In [45]:
print(headlines)

Elon Musk's 2025: From Trump Power Broker To Tesla Turmoil
- Top 5 Tech Regulation Trends To Watch Out For In 2026: AI Governance, Data Privacy And More
- Amazon Permits Remote Work For Employees Stuck In India As Visa Processing Slows: Report
- Gig Work Startup CEO Predicts These Jobs Will Survive Automation In 2026


In [46]:
group['headline'][608]

'Top 5 Tech Regulation Trends To Watch Out For In 2026: AI Governance, Data Privacy And More'

In [ ]:
    bronze_path = f"../../data/bronze/{ticker}/{interval}/data.parquet"
    silver_dir = f"../../data/silver/{ticker}/{interval}"
    silver_path = f"{silver_dir}/data.parquet"

    os.makedirs(silver_dir, exist_ok=True)

    # Make sure we have the bronze data
    if not os.path.exists(bronze_path):
        print(f"[{ticker}] Raw data not found at {bronze_path}. Please run Tier 1 first.")
        return
    
    raw_df = pd.read_parquet(bronze_path, engine='pyarrow')
    raw_df['Date'] = pd.to_datetime(raw_df['Date'])

    # Figure out the required data chunk
    if os.path.exists(silver_path):
        features_df = pd.read_parquet(silver_path, engine='pyarrow')
        features_df['Date'] = pd.to_datetime(features_df['Date'])
        
        # Grab the date of the last calculated feature
        last_feature_date = features_df['Date'].iloc[-1]
        
        # Calculate the exact calendar date we need to prime the math, 
        # accounting for crypto 24/7 vs equity holidays
        prime_date, asset_class = calculate_lookback_date(ticker, last_feature_date, lookback_days)
        
        print(f"[{ticker}] ({asset_class}) Updating features. Priming data from {prime_date.strftime('%Y-%m-%d')}...")
        
        # Slice the raw data from our calculated prime date forward
        processing_df = raw_df[raw_df['Date'] >= prime_date].copy()
        is_incremental = True
        
    else:
        # First time calculating features
        processing_df = raw_df.copy()
        features_df = pd.DataFrame() 
        print(f"[{ticker}] No existing features. Calculating full history...")
        is_incremental = False

    # Compute indicators
    processing_df = apply_indicators(processing_df)

    # Compute and merge the AI Sentiment Feature
    sentiment_df = compute_sentiment_feature(ticker, interval)

    # Merge the sentiment column into your main OHLCV+Indicators dataframe
    processing_df = processing_df.join(sentiment_df, how='left')
    processing_df['Sentiment'] = processing_df['Sentiment'].ffill().fillna(0.0)